# 01. 계층 지도와 첫 호출

LangChain의 공통 모델 인터페이스는 첫 호출의 어느 부분을 맡을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 네 수업 패키지의 검증 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 키 값은 출력하지 않은 채 존재 여부만 확인합니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 제공자·모델과 초 단위 타임아웃, 재시도·출력 상한을 가진 모델 객체를 만듭니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## 메시지와 첫 호출

역할이 있는 메시지 목록을 만들고 `invoke()`를 한 번 호출합니다.

`model`은 호출 설정을 가진 객체, `messages`는 이번 입력, `response`는 완료된 호출의 반환값입니다. `init_chat_model()`은 객체를 준비하고, `model.invoke(messages)`가 질문을 실제 전송합니다. `response.text`는 텍스트를 읽는 속성이므로 뒤에 호출 괄호를 붙이지 않습니다.

메시지는 `(역할, 내용)` 튜플의 목록입니다. 여기서 `system`은 공통 지시이고 `user`는 이번 질문입니다. `[role for role, _ in messages]`는 튜플을 역할과 내용으로 나누어 역할만 모으는 목록 표현식입니다. `_`도 변수 이름이지만 이 자리에서는 내용 값을 사용하지 않는다는 관례로 씁니다.

같은 모델 객체에 다른 질문 목록을 넘길 수 있습니다. 이전 답까지 이어서 대화하려면 그 답을 다음 입력 목록에 직접 포함해야 합니다. 아래의 질문 변경은 같은 호출 방법의 재사용을 확인하며 대화 기억 기능을 시험하지 않습니다.

In [4]:
# messages 목록의 0번 system 규칙을 이후 비교 호출에서도 그대로 재사용합니다.
messages = [
    ("system", "두 문장 이내의 한국어로 답하세요."),
    ("user", "LangChain 모델 인터페이스의 장점을 한 가지 설명하세요."),
]
print("역할:", [role for role, _ in messages])

역할: ['system', 'user']


In [5]:
# invoke 반환값의 text·실제 타입·usage_metadata 존재 여부를 확인합니다.
response = model.invoke(messages)
print(response.text)
print("응답 타입:", type(response).__name__)
print("사용량 있음:", response.usage_metadata is not None)

LangChain의 모델 인터페이스는 다양한 LLM을 동일한 방식으로 연결하고 교체할 수 있어, 코드 변경을 최소화하면서 모델 전환이 쉬운 점이 장점입니다.
응답 타입: AIMessage
사용량 있음: True


## 값 하나 변경

사용자 질문만 바꾸고 같은 모델 객체와 호출 방법을 유지합니다.

In [6]:
# 사용자 질문만 바꾸고 모델 객체와 system 메시지는 그대로 둡니다.
changed = model.invoke([
    messages[0],
    ("user", "LangChain 모델 인터페이스를 한 문장으로 설명하세요."),
])
print(changed.text)
print("실제 반환 타입:", type(changed).__name__)

LangChain 모델 인터페이스는 다양한 LLM과 체인, 도구를 일관된 방식으로 연결·호출할 수 있게 해주는 표준화된 추상화입니다.
실제 반환 타입: AIMessage


## 실패 재현

다음 셀의 `all()`·`isinstance()`·`raise TypeError`는 이 수업에서 직접 작성한 문자열 입력 검사입니다. 실제 API를 호출하기 전에 실행하며 LangChain 내부에서 발생한 예외를 관찰한 결과가 아닙니다. LangChain의 모든 메시지 내용 형식을 문자열로 제한하는 일반 규칙으로 해석하지 않습니다.

In [7]:
# API 요청 전에 content 타입을 검사해 예상한 TypeError만 처리합니다.
bad_messages = [("user", None)]
try:
    if not all(isinstance(content, str) for _, content in bad_messages):
        raise TypeError("메시지 content는 문자열이어야 합니다.")
except TypeError as error:
    print("관찰한 예외 유형:", type(error).__name__)
else:
    raise AssertionError("잘못된 메시지는 예외가 발생해야 합니다.")

관찰한 예외 유형: TypeError


## 수정

In [8]:
# 잘못된 None 하나를 문자열로 고친 뒤 같은 invoke 경로가 성공하는지 확인합니다.
good_messages = [("user", "AIMessage라고만 답하세요.")]
fixed = model.invoke(good_messages)
print("수정 후 응답:", fixed.text)
assert fixed.text.strip()

수정 후 응답: AIMessage
